In [ ]:
import numpy as np
import spikeinterface.full as si
import sys
import os


In [ ]:
file_path = '/mnt/disk20tb/PrimaryNeuronData/Maxtwo/CDKL5-E6D_T1_C1/CDKL5-E6D_T1_C1/240520/M08018/Network/000008/data.raw.h5'

recording1 = si.read_maxwell(file_path,stream_id='well001')

#recording = si.ConcatenateSegmentRecording([recording1,recording2])
channel_ids = recording1.get_channel_ids()
fs = recording1.get_sampling_frequency()
num_chan = recording1.get_num_channels()
num_seg = recording1.get_num_segments()
total_recording = recording1.get_total_duration()

#end_time = int((total_recording-1))
start_time = 0
end_time = 300

#print('Channel ids:', channel_ids)
print('Sampling frequency:', fs)
print('Number of channels:', num_chan)
print('Number of segments:', num_seg)
print(f"total_recording: {total_recording} s")

recording_bp = si.bandpass_filter(recording1, freq_min=300, freq_max=3000)
recording_lfp = si.bandpass_filter(recording1, freq_min=1, freq_max=300)

recodring_cmr = si.common_reference(recording_bp, reference='global', operator='median')
#recording_chunk = recodring_cmr.frame_slice(start_frame= 1*fs,end_frame=425*fs)
recording_chunk = recodring_cmr.frame_slice(start_frame= start_time*fs,end_frame=end_time*fs)
#recording_chunk =si.scale(recording_chunk, gain=3.0)
print(f"chunk duration: {recording_chunk.get_total_duration()} s")

In [ ]:
job_kwargs = dict(n_jobs=32, chunk_duration="1s", progress_bar=True)
recording_chunk.save(folder='/mnt/disk20tb/PrimaryNeuronData/Maxtwo/CDKL5-E6D_T1_C1/CDKL5-E6D_T1_C1/240520/M08018/Network/000008/binary_well1',format='binary',**job_kwargs)

In [ ]:
rec_bin = si.load_extractor('/mnt/disk20tb/PrimaryNeuronData/Maxtwo/CDKL5-E6D_T1_C1/CDKL5-E6D_T1_C1/240604/M07039/Network/000070/binary_well0')

In [ ]:
from scipy.signal import find_peaks

# Get traces from rec_bin
traces = rec_bin.get_traces(segment_index=0)

# Initialize a dictionary to store peak counts for each channel
peak_counts = {}

# Iterate over all channels
for channel_idx in range(traces.shape[1]):
    channel_trace = traces[:, channel_idx]
    peaks, _ = find_peaks(-channel_trace, height=5.5)  # Adjust parameters as needed
    peak_counts[channel_idx] = {  # Initialize the dictionary for the channel
        'spikecounts': len(peaks),
        'spike_indices': peaks.tolist()  # Store the indices of the peaks
    }

# Print peak counts for all channels
print(peak_counts)

In [ ]:
import matplotlib.pyplot as plt

# Sampling rate (in Hz) of the recording
sampling_rate = 10000  # Replace with the actual sampling rate of your data

# Define the time window for the raster plot (first 30 seconds)
time_window = 30  # seconds
max_index = time_window * sampling_rate  # Convert time to sample index

# Initialize a figure for the raster plot
plt.figure(figsize=(12, 8))

# Iterate over all channels in the peak_counts dictionary
for channel_idx, channel_data in peak_counts.items():
    # Filter spike indices to include only those within the first 30 seconds
    filtered_spike_indices = [index for index in channel_data['spike_indices'] if index < max_index]
    
    # Convert spike indices to time (in seconds)
    spike_times = [index / sampling_rate for index in filtered_spike_indices]
    
    # Plot the spikes as dots in the raster plot
    plt.scatter(spike_times, [channel_idx] * len(spike_times), s=1, color='black',marker='|',rasterized=True)  # s=5 sets the dot size

# Customize the plot
plt.title("Raster Plot for the First 30 Seconds")
plt.xlabel("Time (s)")
plt.ylabel("Channel")
#plt.yticks(range(len(peak_counts)))  # Set y-ticks to match channel indices
#plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sampling rate (in Hz) of the recording
sampling_rate = 10000  # Replace with the actual sampling rate of your data

# Define the time window for the raster plot (first 30 seconds)
time_window = 30  # seconds
max_index = time_window * sampling_rate  # Convert time to sample index

# Initialize a figure with two subplots (raster plot and histogram)
fig = plt.figure(figsize=(12, 8))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0)  # No gap between plots

# Histogram (bottom)
ax_hist = fig.add_subplot(gs[1])  # Create the histogram axis first

# Raster plot (top)
ax_raster = fig.add_subplot(gs[0], sharex=ax_hist)  # Share x-axis with the histogram
for channel_idx, channel_data in peak_counts.items():
    # Filter spike indices to include only those within the first 30 seconds
    filtered_spike_indices = [index for index in channel_data['spike_indices'] if index < max_index]
    
    # Convert spike indices to time (in seconds)
    spike_times = [index / sampling_rate for index in filtered_spike_indices]
    
    # Plot the spikes as dots in the raster plot
    ax_raster.scatter(spike_times, [channel_idx] * len(spike_times), s=1, color='royalblue', marker='|', rasterized=True)

# Customize the raster plot
ax_raster.set_title("Raster Plot and Spike Count Histogram")
ax_raster.set_ylabel("Channel")
ax_raster.set_xlim(0, time_window)
ax_raster.tick_params(axis='x', which='both', bottom=False, labelbottom=False)  # Hide x-axis ticks for the raster plot

# Histogram (bottom)
ax_hist = fig.add_subplot(gs[1], sharex=ax_raster)  # Share x-axis with the raster plot

# Combine all spike times across channels for the histogram
all_spike_times = []
for channel_data in peak_counts.values():
    filtered_spike_indices = [index for index in channel_data['spike_indices'] if index < max_index]
    all_spike_times.extend([index / sampling_rate for index in filtered_spike_indices])

# Create a histogram of spike counts over time
bin_width = 0.1  # seconds
bins = np.arange(0, time_window + bin_width, bin_width)
ax_hist.hist(all_spike_times, bins=bins, color='royalblue')

# Customize the histogram
ax_hist.set_xlabel("Time (s)")
ax_hist.set_ylabel("Spike Count")
ax_hist.set_xlim(0, time_window)

# Show the combined figure
plt.tight_layout()
plt.show()

In [ ]:
file_path = '/mnt/disk20tb/PrimaryNeuronData/Maxtwo/CDKL5-E6D_T1_C1/CDKL5-E6D_T1_C1/240604/M07039/Network/000070/data.raw.h5'

recording1 = si.read_maxwell(file_path,stream_id='well000')

#recording = si.ConcatenateSegmentRecording([recording1,recording2])
channel_ids = recording1.get_channel_ids()
fs = recording1.get_sampling_frequency()
num_chan = recording1.get_num_channels()
num_seg = recording1.get_num_segments()
total_recording = recording1.get_total_duration()

#end_time = int((total_recording-1))
start_time = 0
end_time = 300

#print('Channel ids:', channel_ids)
print('Sampling frequency:', fs)
print('Number of channels:', num_chan)
print('Number of segments:', num_seg)
print(f"total_recording: {total_recording} s")

recording_bp = si.bandpass_filter(recording1, freq_min=300, freq_max=3000)
recording_lfp = si.bandpass_filter(recording1, freq_min=1, freq_max=300)

recodring_cmr = si.common_reference(recording_bp, reference='global', operator='median')
#recording_chunk = recodring_cmr.frame_slice(start_frame= 1*fs,end_frame=425*fs)
recording_chunk = recodring_cmr.frame_slice(start_frame= start_time*fs,end_frame=end_time*fs)
#recording_chunk =si.scale(recording_chunk, gain=3.0)
print(f"chunk duration: {recording_chunk.get_total_duration()} s")


